In [26]:
# Sabse pehle faker library install karein
!pip install -q faker

import pandas as pd
import numpy as np
from faker import Faker
import random
import warnings

# Saari faltu red warnings ko chupane ke liye
warnings.filterwarnings('ignore')

fake = Faker()
np.random.seed(42)
random.seed(42)

num_records = 1500
data = []
cities = ['Karachi', 'Lahore', 'Islamabad', 'Rawalpindi', 'Faisalabad']
subscriptions = ['Basic', 'Standard', 'Premium']

for i in range(num_records):
    customer_id = f"CUST-{1000 + i}"
    age = random.randint(18, 70)
    gender = random.choice(['Male', 'Female'])
    city = random.choice(cities)
    subscription_type = random.choice(subscriptions)
    tenure = random.randint(1, 60)
    monthly_spending = round(random.uniform(20.0, 200.0), 2)
    num_purchases = random.randint(1, 15)
    support_requests = random.randint(0, 9)
    login_frequency = random.randint(1, 30)
    satisfaction_score = random.randint(1, 5)

    churn_probability = 0.1
    if support_requests > 5: churn_probability += 0.3
    if satisfaction_score <= 2: churn_probability += 0.4
    if tenure < 6: churn_probability += 0.1

    churn_status = 1 if random.random() < churn_probability else 0
    data.append([customer_id, age, gender, city, subscription_type, monthly_spending, tenure, num_purchases, support_requests, login_frequency, satisfaction_score, churn_status])

columns = ['CustomerID', 'Age', 'Gender', 'City', 'SubscriptionType', 'MonthlySpending', 'Tenure', 'NumberOfPurchases', 'CustomerSupportRequests', 'LoginFrequency', 'SatisfactionScore', 'ChurnStatus']
df = pd.DataFrame(data, columns=columns)
print("--- Cell 1: Data Successfully Created! ---")

--- Cell 1: Data Successfully Created! ---


In [27]:
# CustomerID drop karna aur categorical text ko numbers mein badalna
df = df.drop(columns=['CustomerID'])
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})
df = pd.get_dummies(df, columns=['City', 'SubscriptionType'], drop_first=True)
print("--- Cell 2: Data Successfully Encoded! ---")

--- Cell 2: Data Successfully Encoded! ---


In [28]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# Features aur Target alag karein
X = df.drop(columns=['ChurnStatus'])
y = df['ChurnStatus']

# Split aur Scale
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Model Training
rf_model = RandomForestClassifier(random_state=42, n_estimators=100)
rf_model.fit(X_train_scaled, y_train)
print("--- Cell 3: Model Successfully Trained! ---")

--- Cell 3: Model Successfully Trained! ---


In [29]:
import pickle

with open('churn_model.pkl', 'wb') as model_file:
    pickle.dump(rf_model, model_file)

with open('scaler.pkl', 'wb') as scaler_file:
    pickle.dump(scaler, scaler_file)

print("--- Cell 4: Model and Scaler Saved! ---")

--- Cell 4: Model and Scaler Saved! ---


In [30]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import pickle

# Load the saved model and scaler
with open('churn_model.pkl', 'rb') as f:
    model = pickle.load(f)

with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

# Web App Dashboard Title and Description
st.title("📊 Customer Churn Prediction Dashboard")
st.write("Enter the customer details below to predict the risk of churn.")

# User Inputs (English Labels)
age = st.slider("Customer Age", 18, 70, 35)
gender = st.selectbox("Customer Gender", ["Male", "Female"])
tenure = st.slider("Tenure (Months)", 1, 60, 12)
monthly_spending = st.number_input("Monthly Spending ($)", 20.0, 200.0, 50.0)
purchases = st.slider("Number of Purchases", 1, 15, 5)
support_requests = st.slider("Customer Support Requests", 0, 9, 2)
login_freq = st.slider("Login Frequency (Per Month)", 1, 30, 15)
satisfaction = st.slider("Customer Satisfaction Score (1-5)", 1, 5, 4)

# Dummy features to match dataset shape
city_and_sub_inputs = [0, 0, 0, 0, 0, 0]

# Prediction Button
if st.button("Predict Churn Risk"):
    gender_encoded = 1 if gender == "Male" else 0

    # Structure input data
    input_data = [age, gender_encoded, monthly_spending, tenure, purchases, support_requests, login_freq, satisfaction] + city_and_sub_inputs
    input_scaled = scaler.transform([input_data])
    prediction = model.predict(input_scaled)

    st.subheader("Prediction Result:")
    if prediction[0] == 1:
        st.error("⚠️ High Churn Risk! This customer is likely to leave the business.")
    else:
        st.success("✅ Low Churn Risk! This customer is safe and likely to stay.")

Overwriting app.py


In [34]:
from google.colab import output
output.eval_js('google.colab.kernel.proxyPort(8501)')

'https://8501-m-s-kkb-use4a2-1lhti2vv8ddfk-a.us-east4-2.prod.colab.dev'

In [ ]:
# 1. Pehle saare phanse hue processes ka khatma
!pkill -f streamlit
!pkill -f lt

# 2. Node.js ka localtunnel install karein (agar pehle miss ho gaya ho)
!npm install -g localtunnel -q

# 3. Streamlit ko background mein chalaein
get_ipython().system_raw('streamlit run app.py --server.port 8501 &')

# 4. Colab ka public IP nikalein (Yeh password ke taur par kaam aayega)
print("------------------------------------------------------------")
print("🔑 APKA TUNNEL PASSWORD NEECHE HAI (Isko copy kar lein):")
!curl ipv4.icanhazip.com
print("------------------------------------------------------------")

import time
time.sleep(5) # Server ko active hone ka poora waqt dein

# 5. Localtunnel ko secure port 8501 par chalaein
print("🔥 APKA LIVE INTERFACE LINK TAYAR HO RAHA HAI... 👇")
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
added 22 packages in 4s
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼------------------------------------------------------------
🔑 APKA TUNNEL PASSWORD NEECHE HAI (Isko copy kar lein):
8.234.150.41
------------------------------------------------------------
🔥 APKA LIVE INTERFACE LINK TAYAR HO RAHA HAI... 👇
⠙⠹⠸⠼⠴your url is: https://bumpy-olives-cheer.loca.lt
